In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from tqdm import tqdm
from sklearn.feature_extraction.text import CountVectorizer

In [2]:
shapefiles_path = "../../data/final/shapefiles/"
keywords_and_location_names_path = "../../data/final/keywords_and_location_names/"

counts_path = "../../data/final/keyword_location_counts/"

newsapi_downloads_path = "../../data/final/downloads_newsapi/"

In [ ]:
english_keywords_dict = pd.read_pickle(keywords_and_location_names_path + 'id_english_keyword.pkl')
arabic_keywords_dict = pd.read_pickle(keywords_and_location_names_path + 'id_arabic_keyword.pkl')

In [ ]:
english_location_names_dict = pd.read_pickle(keywords_and_location_names_path + 'id_english_location_name.pkl')
arabic_location_names_dict = pd.read_pickle(keywords_and_location_names_path + 'id_arabic_location_name.pkl')

# 1. Associating keywords and locations with articles

## 1.1 Basic functions

In [85]:
def get_strings_and_column_names(dictionary):
    ids = [key for key in dictionary.keys()]
    names = np.concatenate([dictionary[id] for id in ids])
    return list(names)

In [86]:
def create_keyword_location_count_dataframe(df, keyword_dict, location_dict):
    # Create columns for the different keyword IDs by summing over all the columns 
    # containing words representing the same keyword
    keyword_id_column_list = []
    for keyword_id in keyword_dict.keys():
        keyword_id_column_list.append(df[keyword_dict[keyword_id]].sum(axis=1))
        
    # Sum over the counts of all possible variants of how the location names are written
    location_id_column_list = []
    for key in location_dict.keys():
        location_id_column_list.append(df[location_dict[key]].sum(axis=1))
        
    column_list = keyword_id_column_list + location_id_column_list
    location_keyword_df = pd.concat(column_list, axis=1)
    location_keyword_df.columns = np.concatenate([list(keyword_dict.keys()), list(location_dict.keys())])
    
    return location_keyword_df

## 1.2 Processing English Articles

In [1]:
download_file_specification = "Mashreq_2024-06-23_2024-07-24"

In [4]:
newsapi_downloads_path + download_file_specification + "_articles_eng.csv"

'../../data/final/downloads_newsapi/Mashreq_2024-06-23_2024-07-24_articles_eng.csv'

In [88]:
# Loading the raw downloaded data
eng = pd.read_csv(newsapi_downloads_path + download_file_specification + "_articles_eng.csv") 
eng = eng.drop(columns=['userHasPermissions'])

In [89]:
# Adding the body length columnS
eng["body_len"] = eng["body"].apply(lambda x: len(x))
eng["body_len_str"] = eng["body"].apply(lambda x: len(x.split(" ")))

# Converting the article body to lowercase
eng["body"] = eng["body"].apply(lambda x: x.lower())

In [90]:
keyword_names_eng = get_strings_and_column_names(english_keywords_dict)
location_names_eng = get_strings_and_column_names(english_location_names_dict)
vocabulary_eng = np.unique(np.concatenate([keyword_names_eng, location_names_eng]))

In [91]:
# The CountVectorizer requires as input the numbers of ngrams to consider, so we need to find the maximum number of ngrams required
ngrams_upper_bound_eng = np.max([len(word.split(" ")) for word in vocabulary_eng])
print(f"The upper bound for the n-grams is {ngrams_upper_bound_eng}")

The upper bound for the n-grams is 4


In [92]:
count_eng = CountVectorizer(vocabulary=vocabulary_eng, ngram_range=(1, ngrams_upper_bound_eng)).fit_transform(eng["body"].values).toarray()
df_eng = pd.DataFrame(count_eng, columns=vocabulary_eng)

In [93]:
location_keyword_df_eng = create_keyword_location_count_dataframe(df_eng, english_keywords_dict, english_location_names_dict)

In [94]:
# Merge the article data with the keyword counts
eng = eng.merge(location_keyword_df_eng, left_index=True, right_index=True)

# Add a column that sums over the mentions of all keywords
eng["kw_all"] = eng[list(english_keywords_dict.keys())].sum(axis=1)

In [95]:
eng.to_csv(counts_path + download_file_specification + "_eng_keywords_location_counts.csv", index=False)

## 1.3 Processing Arabic Articles

In [96]:
# Loading the raw downloaded data
ara = pd.read_csv(newsapi_downloads_path + download_file_specification + "_articles_ara.csv")

# Adding the body length column
ara["body_len"] = ara["body"].apply(lambda x: len(x))
ara["body_len_str"] = ara["body"].apply(lambda x: len(x.split(" ")))

# Note that Arabic text has only one case, so we don't need to convert it to lowercase

In [97]:
keyword_names_ara = get_strings_and_column_names(arabic_keywords_dict)
location_names_ara = get_strings_and_column_names(arabic_location_names_dict)
vocabulary_ara = np.unique(np.concatenate([keyword_names_ara, location_names_ara]))

In [98]:
# The CountVectorizer requires as input the numbers of ngrams to consider, so we need to find the maximum number of ngrams required
ngrams_upper_bound_ara = np.max([len(word.split(" ")) for word in vocabulary_ara])
print(f"The upper bound for the n-grams is {ngrams_upper_bound_ara}")

The upper bound for the n-grams is 6


In [99]:
count_ara = CountVectorizer(vocabulary=vocabulary_ara, ngram_range=(1, ngrams_upper_bound_ara)).fit_transform(ara["body"].values).toarray()
df_ara = pd.DataFrame(count_ara, columns=vocabulary_ara)

In [101]:
location_keyword_df_ara = create_keyword_location_count_dataframe(df_ara, arabic_keywords_dict, arabic_location_names_dict)

In [102]:
# Merge the article data with the keyword counts
ara = ara.merge(location_keyword_df_ara, left_index=True, right_index=True)

# Add a column that sums over the mentions of all keywords
ara["kw_all"] = ara[list(arabic_keywords_dict.keys())].sum(axis=1)

In [103]:
ara.to_csv(counts_path + download_file_specification + "_ara_keywords_location_counts.csv", index=False)

# 2. Create summary table

The summary dictionaries count the number of articles that mention at least one keyword of a certain keyword category.
This means that also if multiple keywords of a category are mentioned in an article or an article mentions the same keyword multiple times, 
it only counts as one in the column representing the corresponding keyword category. 

However, an article can be represented with a one in multiple category columns, which is why the sum over all category columns does not correspond to the number of articles mentioning any keyword.
This count is represented by the "kw" column, which is therefore always smaller or equal to the sum of the columns of the different keyword categories. 

If an article mentions multiple locations, it also appears acordingly in multiple rows represented as a one

## 2.1 Basic Functions

In [ ]:
def create_keyword_to_category_dict(keywords_dict:dict) -> dict:
    """
    Create a dictionary mapping keyword categories to their corresponding keywords.
    Adding a keyword category "kw" containing as keyword ["kw_all"].
    
    Args:
        keywords_dict (dict): A dictionary containing keywords as keys.
        The keys must have the format "category_keyword".
    
    Returns:
        dict: A dictionary mapping keyword categories to lists of keywords.
    """
    keyword_to_category_dict = {}

    keyword_categories = np.unique([key.split("_")[0] for key in keywords_dict.keys()])
    for category in keyword_categories:
        keyword_to_category_dict[category] = [key for key in keywords_dict.keys() if key.startswith(category + "_")]
        
    keyword_to_category_dict["kw"] = ["kw_all"]
    
    return keyword_to_category_dict

In [33]:
def create_summary_df(news_articles_with_keyword_counts: pd.DataFrame, keyword_to_category_dict: dict, location_names_dict: dict) -> pd.DataFrame:
    """
    Create a summary dataframe of daily counts of articles mentioning keywords.
    Important: The output dataframe counts articles mentioning keywords and not keywords themselves.
    
    Args:
        news_articles_with_keyword_counts (pd.DataFrame): A dataframe containing the counts of articles mentioning keywords.
        keyword_to_category_dict (dict): A dictionary mapping keyword group codes to their corresponding column names.
        location_names_dict (dict): A dictionary mapping location IDs to their corresponding names.
    
    Returns:
        pd.DataFrame: A summary dataframe with daily counts of articles mentioning keywords per location and keyword group.
    """
    
    # Get all dates from the language dataframe
    unique_dates = news_articles_with_keyword_counts["date"].sort_values().unique()

    date_dfs = []
    
    # Iterate over all country codes
    for location_id in tqdm(location_names_dict.keys()):
                                
        # Create a dataframe with all unique dates, province names and country names
        date_df = pd.DataFrame(data={"date":unique_dates})
        date_df["location"] = location_id
            
        # Count the number of articles mentioning a certain province for each date, name the columns of this dataframe "date" and "count_articles"
        no_articles = news_articles_with_keyword_counts.loc[(news_articles_with_keyword_counts[location_id] > 0),].groupby("date").size().reset_index()
        no_articles.columns = ["date", "count_articles"]
        
        # Merge the count information with the date dataframe
        date_df = date_df.merge(no_articles, on="date", how="left")
        
        # Iterate over the Keyword Groups
        for keyword_group_code in list(keyword_to_category_dict.keys()):

            # Extract the column names for all columns of the keyword group
            keyword_group_columns = keyword_to_category_dict[keyword_group_code]

            # Count the number of articles mentioning a certain province and a certain keyword group for each date, name the columns of this dataframe "date" and the keyword group code
            date_count_df = news_articles_with_keyword_counts.loc[(news_articles_with_keyword_counts[location_id] > 0) & (news_articles_with_keyword_counts[keyword_group_columns].sum(axis=1) > 0),].groupby("date")[keyword_group_columns].count().iloc[:,0]
            date_count_df = pd.DataFrame(date_count_df).reset_index()
            date_count_df.columns = ["date", keyword_group_code]
            
            # Merge the count information with the date dataframe
            date_df = date_df.merge(date_count_df, on="date", how="left")
            
            for keyword_group_column in keyword_group_columns:
                date_count_df = news_articles_with_keyword_counts.loc[(news_articles_with_keyword_counts[location_id] > 0) & (news_articles_with_keyword_counts[keyword_group_column] > 0),].groupby("date")[keyword_group_column].count()
                date_count_df = pd.DataFrame(date_count_df).reset_index()
                date_count_df.columns = ["date", keyword_group_column]
                
                # Merge the count information with the date dataframe
                date_df = date_df.merge(date_count_df, on="date", how="left")
                   
        date_dfs.append(date_df)
            
    # Concatenate the date dataframes for all the provinces
    daily_count_of_articles_mentioning_keyword_per_location = pd.concat(date_dfs).reset_index(drop=True)
    daily_count_of_articles_mentioning_keyword_per_location["date"] = pd.to_datetime(daily_count_of_articles_mentioning_keyword_per_location["date"])
    daily_count_of_articles_mentioning_keyword_per_location.set_index("date", inplace=True)

    daily_count_of_articles_mentioning_keyword_per_location.drop(columns="kw_all", inplace=True)
    return daily_count_of_articles_mentioning_keyword_per_location

## 2.2 English

In [ ]:
keywords_dict = english_keywords_dict
location_names_dict = english_location_names_dict
news_articles_with_keyword_counts = pd.read_csv("../../data/final/keyword_location_counts/Mashreq_2024-06-23_2024-07-24_articles_eng_counts.csv")
language = "English"

In [ ]:
# Create a dictionary mapping keyword categories to their corresponding keywords
keyword_to_category_dict = create_keyword_to_category_dict(keywords_dict)

In [ ]:
# Create a summary dataframe with daily counts of articles mentioning keywords per location
daily_count_of_articles_mentioning_keyword_per_location_eng = create_summary_df(news_articles_with_keyword_counts, keyword_to_category_dict, location_names_dict)

In [ ]:
# Fill missing values with 0
daily_count_of_articles_mentioning_keyword_per_location_eng = daily_count_of_articles_mentioning_keyword_per_location_eng.fillna(0)

In [ ]:
# Create a new admin level column indicating the level of the administrative division (0: Country, 1: Province, 2: District)
daily_count_of_articles_mentioning_keyword_per_location_eng.insert(2, "admin_level", 0, allow_duplicates=False)
daily_count_of_articles_mentioning_keyword_per_location_eng["admin_level"] = daily_count_of_articles_mentioning_keyword_per_location_eng["location"].str.split("_").apply(lambda x: len(x)) -1

In [ ]:
# Insert a column for the language
daily_count_of_articles_mentioning_keyword_per_location_eng.insert(3, "language", language, allow_duplicates=False)

## 2.3 Arabic

In [ ]:
keywords_dict = arabic_keywords_dict
location_names_dict = arabic_location_names_dict
news_articles_with_keyword_counts = pd.read_csv("../../data/final/keyword_location_counts/Mashreq_2024-06-23_2024-07-24_articles_ara_counts.csv")
language = "English"

In [ ]:
# Create a dictionary mapping keyword categories to their corresponding keywords
keyword_to_category_dict = create_keyword_to_category_dict(keywords_dict)

In [ ]:
# Create a summary dataframe with daily counts of articles mentioning keywords per location
daily_count_of_articles_mentioning_keyword_per_location_ara = create_summary_df(news_articles_with_keyword_counts, keyword_to_category_dict, location_names_dict)

In [ ]:
# Fill missing values with 0
daily_count_of_articles_mentioning_keyword_per_location_ara = daily_count_of_articles_mentioning_keyword_per_location_ara.fillna(0)

In [ ]:
# Create a new admin level column indicating the level of the administrative division (0: Country, 1: Province, 2: District)
daily_count_of_articles_mentioning_keyword_per_location_ara.insert(2, "admin_level", 0, allow_duplicates=False)
daily_count_of_articles_mentioning_keyword_per_location_ara["admin_level"] = daily_count_of_articles_mentioning_keyword_per_location_ara["location"].str.split("_").apply(lambda x: len(x)) -1

In [ ]:
# Insert a column for the language
daily_count_of_articles_mentioning_keyword_per_location_ara.insert(3, "language", language, allow_duplicates=False)

## 2.4 Joining the two dataframes

In [188]:
daily_count_of_articles_mentioning_keyword_per_location = pd.concat([daily_count_of_articles_mentioning_keyword_per_location_eng, daily_count_of_articles_mentioning_keyword_per_location_ara])
daily_count_of_articles_mentioning_keyword_per_location.reset_index(inplace=True)

In [196]:
#daily_count_of_articles_mentioning_keyword_per_location.to_csv(data_folder + "/newsapi/summary-dataframes/summary_df_2024_06_23_2024_07_24.csv", index=False)